In [4]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [5]:
import matplotlib.pyplot as plt
import seaborn as sns

# Global seaborn / matplotlib defaults
sns.set_theme(
    style="whitegrid",          # axes with grid
    # palette="colorblind",            # default colour palette
    # font_scale=1.3,             # scales all font sizes uniformly
    rc={
        # "lines.linewidth": 4,           # default line width
        # "axes.titlesize": 16,
        # "axes.labelsize": 22,
        # "axes.labelweight": "bold",     
        # "xtick.labelsize": 18,
        # "ytick.labelsize": 18,
        # "legend.fontsize": 19,
        "grid.linestyle": "-",
        "grid.alpha": 0.6,
        # "lines.markersize": 8,

    },
)

Load data

In [6]:
import pandas as pd


def preview_results(df: pd.DataFrame, sample_size: int = 10) -> None:
    if len(df) > 0:
        display(df.sample(min(sample_size, len(df))))


logprob_results_path = "../analysis/results/logprob_acc_merged.json"
logprob_results = pd.read_json(logprob_results_path, orient="records")

logprob_norm_results_path = "../analysis/results/logprob_acc_norm_merged.json"
logprob_norm_results = pd.read_json(logprob_norm_results_path, orient="records")

generative_results_path = "../analysis/generative_tails.json"
generative_results = pd.read_json(generative_results_path, orient="records")

In [8]:
raw_results_path = "../logs/silent-norm-final-v1/results.json"
raw_results = pd.read_json(raw_results_path, orient="records")

In [9]:
def add_path_metadata(dirty_df: pd.DataFrame) -> pd.DataFrame:
    dirty_df = dirty_df.copy()
    dirty_parts = dirty_df["path"].str.split("/")
    if (dirty_parts.str.len() < 2).any():
        raise ValueError("Clean path does not contain enough segments to parse model_name")

    dirty_out = dirty_df.copy()

    dirty_out["model_name"] = dirty_parts.str[-5]
    dirty_out["train_dataset"] = dirty_parts.str[-4]
    dirty_out["layer_name"] = dirty_parts.str[-3]
    dirty_out["exp_name"] = dirty_parts.str[-2]

    return dirty_out


# apply formatting to metric column
def format_metric_column(df: pd.DataFrame, metric_col: str = "metric") -> pd.DataFrame:
    df = df.copy()
    df[metric_col] = df[metric_col].apply(lambda x: x.replace(",none", ""))
    return df

raw_results = add_path_metadata(raw_results)
raw_results = format_metric_column(raw_results)
raw_results

,path,file,benchmark,metric,value,model_name,train_dataset,layer_name,exp_name
0,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r1,acc,0.349000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
1,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r2,acc,0.325000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
2,/home/fre.gilad/source/silent_direction/logs/s...,anli.json,anli_r3,acc,0.359167,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
3,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp,acc,0.763642,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
4,/home/fre.gilad/source/silent_direction/logs/s...,blimp.json,blimp_adjunct_island,acc,0.836000,Llama-2-7b-chat-hf,oasst2_tulu-v3,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1
...,...,...,...,...,...,...,...,...,...
27106,/home/fre.gilad/source/silent_direction/logs/s...,xquad_es.json,xquad_es,f1,0.194621,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1
27107,/home/fre.gilad/source/silent_direction/logs/s...,xquad_ru.json,xquad_ru,exact_match,0.015126,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1
27108,/home/fre.gilad/source/silent_direction/logs/s...,xquad_ru.json,xquad_ru,f1,0.169717,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1
27109,/home/fre.gilad/source/silent_direction/logs/s...,xquad_zh.json,xquad_zh,exact_match,0.001681,gemma-2b-it,oasst2_tulu-v3,model.norm,gemma-2b-it-KL-8.0-iter1


In [10]:
def kl_val(row: pd.Series) -> float:
    # Llama-2-7b-chat-hf-KL-0.0-iter1
    exp_name = row["exp_name"]
    kl_str = exp_name.split("KL-")[-1].split("-")[0]
    return float(kl_str)

logprob_results["kl"] = logprob_results.apply(kl_val, axis=1)
logprob_norm_results["kl"] = logprob_norm_results.apply(kl_val, axis=1)
generative_results["kl"] = generative_results.apply(kl_val, axis=1)

raw_results["kl"] = raw_results.apply(kl_val, axis=1)

In [11]:
def run_id(df: pd.DataFrame, id_cols:list[str]) -> pd.Series:
    return df[id_cols].astype(str).agg("-".join, axis=1)

run_id_cols = ["model_name", "layer_name", "exp_name"]

logprob_results["run_id"] = run_id(logprob_results, run_id_cols)
logprob_norm_results["run_id"] = run_id(logprob_norm_results, run_id_cols)
generative_results["run_id"] = run_id(generative_results, run_id_cols)

raw_results["run_id"] = run_id(raw_results, run_id_cols)

Process Raw Results

In [12]:
raw_results['benchmark_metric'] = raw_results['benchmark'] + "/" + raw_results['metric'] 

In [13]:
raw_results.columns.tolist()
pivot_raw_results = raw_results.pivot_table(index=["model_name", "layer_name", 'kl'], columns="benchmark_metric", values="value")


In [14]:
spoadkfnakosjdlf = 15 + 50
pivot_raw_results['global/kl_div'] = (15/spoadkfnakosjdlf) * pivot_raw_results['eval-oasst2/kl_div'] + (50/spoadkfnakosjdlf) * pivot_raw_results['eval-tulu-v3/kl_div']
pivot_raw_results['global/proj_l2_rel'] = (15/spoadkfnakosjdlf) * pivot_raw_results['eval-oasst2/proj_l2_rel'] + (50/spoadkfnakosjdlf) * pivot_raw_results['eval-tulu-v3/proj_l2_rel']


In [15]:
# Get the reference values where kl == 0
ref_values = pivot_raw_results.xs(0.0, level='kl')[['global/kl_div', 'global/proj_l2_rel']]

# Join reference values to the pivot table
# We don't need to manually initialize the _max columns as the join will create them
if 'global/kl_div_max' in pivot_raw_results.columns:
    pivot_raw_results = pivot_raw_results.drop(columns=['global/kl_div_max', 'global/proj_l2_rel_max'])

pivot_raw_results = pivot_raw_results.join(ref_values, on=['model_name', 'layer_name'], rsuffix='_max')

pivot_raw_results = pivot_raw_results[['global/kl_div', 'global/kl_div_max', 'global/proj_l2_rel', 'global/proj_l2_rel_max']]

Concat logprob with logprob results

In [16]:
logprob_results['metric'] = "acc"
logprob_norm_results['metric'] = "acc_norm"
generative_results['metric'] = "generative"

In [17]:
# add logprob_norm_results to logprob_results
logprob_results = pd.concat([logprob_results, logprob_norm_results], ignore_index=True)
logprob_results

,benchmark,model_name,layer_name,exp_name,clean_mean,dirty_mean,clean_std,dirty_std,two_sided_tail,lower_tail,upper_tail,count,diff_prob,choice_metric,kl,run_id,metric
0,anli_r1,Llama-2-7b-chat-hf,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.401416,0.350160,0.008166,0.011055,2.220916e-04,1.139196e-04,1.081719e-04,1000,0.000525,7.475083e-04,0.0,Llama-2-7b-chat-hf-model.embed_tokens-Llama-2-...,acc
1,anli_r2,Llama-2-7b-chat-hf,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.391547,0.350911,0.008316,0.010933,3.145660e-03,1.741479e-03,1.404181e-03,1000,0.009762,1.290762e-02,0.0,Llama-2-7b-chat-hf-model.embed_tokens-Llama-2-...,acc
2,anli_r3,Llama-2-7b-chat-hf,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.371106,0.361384,0.007587,0.011175,2.919351e-01,1.459817e-01,1.459534e-01,1200,0.663413,9.553478e-01,0.0,Llama-2-7b-chat-hf-model.embed_tokens-Llama-2-...,acc
3,mastermind_24_easy,Llama-2-7b-chat-hf,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.327836,0.246694,0.008346,0.007161,0.000000e+00,0.000000e+00,0.000000e+00,1522,0.000000,0.000000e+00,0.0,Llama-2-7b-chat-hf-model.embed_tokens-Llama-2-...,acc
4,mastermind_35_easy,Llama-2-7b-chat-hf,model.embed_tokens,Llama-2-7b-chat-hf-KL-0.0-iter1,0.469538,0.220347,0.006696,0.005856,2.220446e-16,6.592567e-172,2.220446e-16,1856,0.000000,2.220446e-16,0.0,Llama-2-7b-chat-hf-model.embed_tokens-Llama-2-...,acc
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6829,blimp,gemma-2b-it,model.norm,gemma-2b-it-KL-0.5-iter1,0.524923,0.525487,0.001917,0.002711,3.862504e-01,1.929640e-01,1.932864e-01,33500,1.000000,1.386250e+00,0.5,gemma-2b-it-model.norm-gemma-2b-it-KL-0.5-iter1,acc_norm
6830,blimp,gemma-2b-it,model.norm,gemma-2b-it-KL-1.0-iter1,0.524923,0.525255,0.001917,0.002711,3.904710e-01,1.950898e-01,1.953811e-01,33500,1.000000,1.390471e+00,1.0,gemma-2b-it-model.norm-gemma-2b-it-KL-1.0-iter1,acc_norm
6831,blimp,gemma-2b-it,model.norm,gemma-2b-it-KL-2.0-iter1,0.524923,0.525545,0.001917,0.002710,3.848854e-01,1.922778e-01,1.926076e-01,33500,1.000000,1.384885e+00,2.0,gemma-2b-it-model.norm-gemma-2b-it-KL-2.0-iter1,acc_norm
6832,blimp,gemma-2b-it,model.norm,gemma-2b-it-KL-4.0-iter1,0.524923,0.528749,0.001917,0.002706,1.837224e-01,9.166953e-02,9.205285e-02,33500,0.999982,1.183704e+00,4.0,gemma-2b-it-model.norm-gemma-2b-it-KL-4.0-iter1,acc_norm


In [18]:
merge_keys = ["model_name", "layer_name", "kl"]

logprob_results = logprob_results.merge(
    pivot_raw_results.reset_index(),
    on=merge_keys,
    how="left",
)

generative_results = generative_results.merge(
    pivot_raw_results.reset_index(),
    on=merge_keys,
    how="left",
)

In [19]:
def abs_diff(df: pd.DataFrame, col1: str, col2: str, new_col: str="abs_diff") -> pd.DataFrame:
    df[new_col] = (df[col1] - df[col2]).abs()
    return df

# add absolute difference between dirty_mean and clean_mean
logprob_results = abs_diff(logprob_results, "dirty_mean", "clean_mean", "abs_diff")
logprob_norm_results = abs_diff(logprob_norm_results, "dirty_mean", "clean_mean", "abs_diff")
generative_results = abs_diff(generative_results, "value", "clean_mean", "abs_diff")

In [20]:
def abs_diff(df: pd.DataFrame, col1: str, col2: str, new_col: str="diff") -> pd.DataFrame:
    df[new_col] = (df[col1] - df[col2])
    return df

# add absolute difference between dirty_mean and clean_mean
logprob_results = abs_diff(logprob_results, "dirty_mean", "clean_mean", "diff")
logprob_norm_results = abs_diff(logprob_norm_results, "dirty_mean", "clean_mean", "diff")
generative_results = abs_diff(generative_results, "value", "clean_mean", "diff")

In [21]:
threshold = 0.015
generative_results['diff_prob'] = (((generative_results['value'] - generative_results['clean_mean']).abs()) < threshold).astype(float)

In [22]:
choice_metric_col = "silence_score"

In [23]:
logprob_results[choice_metric_col] = logprob_results['lower_tail'] + logprob_results['diff_prob']
generative_results[choice_metric_col] = generative_results['lower_tail'] + generative_results['diff_prob']

In [24]:
logprob_results['diff_prob'].describe()

count    6834.000000
mean        0.620582
std         0.316642
min         0.000000
25%         0.335946
50%         0.722312
75%         0.888569
max         1.000000
Name: diff_prob, dtype: float64

In [25]:
# mask = generative_results['abs_diff'] < 0.015
# a=generative_results[mask].groupby(['model_name', 'layer_name', 'kl'])['two_sided_tail'].describe().reset_index()

In [26]:
mask_logprob = logprob_results['diff'] <= 0.0
logprob_results = logprob_results[mask_logprob]

mask_generative = generative_results['diff'] <= 0.0
generative_results = generative_results[mask_generative]

## Now AGREGRATE

In [27]:
def agg_results(
    df: pd.DataFrame,
    prob_col: str = "two_sided_tail",
    new_col: str = "two_sided_tail_agg",
    group_by_cols: list[str] | None = None,
    keep_cols: list[str] | None = None,
) -> pd.DataFrame:
    keep_cols_local = list(keep_cols) if keep_cols is not None else []
    keep_cols_local = list(dict.fromkeys([*keep_cols_local, prob_col]))  # unique, order-preserving

    group_cols = list(group_by_cols) if group_by_cols is not None else []
    selected_cols = list(dict.fromkeys([*group_cols, *keep_cols_local]))

    if group_by_cols is not None:
        idx = df.groupby(group_by_cols, sort=False)[prob_col].idxmin()
        agg_df = df.loc[idx, selected_cols].reset_index(drop=True)
    else:
        idx = df[prob_col].idxmin()
        agg_df = df.loc[[idx], selected_cols].reset_index(drop=True)

    agg_df = agg_df.rename(columns={prob_col: new_col})
    return agg_df

In [28]:
raw_keep_cols = ['global/kl_div', 'global/kl_div_max', 'global/proj_l2_rel', 'global/proj_l2_rel_max', 'diff_prob', 'two_sided_tail', 'lower_tail', 'upper_tail', 'silence_score']
agg_choice_metric_col = f"agg_{choice_metric_col}"

In [29]:
abs_diff_threshold = - 1e-10

agg_logprobs =  agg_results(
    logprob_results.copy(),
    # threshold=abs_diff_threshold,
    # diff_col="abs_diff",
    prob_col=choice_metric_col,
    new_col=agg_choice_metric_col,
    group_by_cols=["model_name", "layer_name", "kl"],
    keep_cols=raw_keep_cols,
)

agg_generative =  agg_results(
    generative_results.copy(),
    # threshold=abs_diff_threshold,
    # diff_col="abs_diff",
    prob_col=choice_metric_col,
    new_col=agg_choice_metric_col,
    group_by_cols=["model_name", "layer_name", "kl"],
    keep_cols=raw_keep_cols,
)

In [30]:
agg_generative.sort_values("agg_silence_score", ascending=False)

,model_name,layer_name,kl,global/kl_div,global/kl_div_max,global/proj_l2_rel,global/proj_l2_rel_max,diff_prob,two_sided_tail,lower_tail,upper_tail,agg_silence_score
36,Llama-2-7b-chat-hf,model.norm,1.00,0.002950,0.632451,0.092323,0.101190,1.0,0.422650,0.211325,0.211325,1.211325
35,Llama-2-7b-chat-hf,model.norm,0.50,0.005384,0.632451,0.094495,0.101190,1.0,0.422650,0.211325,0.211325,1.211325
39,Llama-2-7b-chat-hf,model.norm,8.00,0.001279,0.632451,0.084812,0.101190,1.0,0.422650,0.211325,0.211325,1.211325
37,Llama-2-7b-chat-hf,model.norm,2.00,0.002157,0.632451,0.090138,0.101190,1.0,0.422650,0.211325,0.211325,1.211325
149,Qwen2.5-3B-Instruct,model.layers.24,8.00,0.008496,2.623426,0.000389,0.372008,1.0,0.129612,0.064806,0.064806,1.064806
...,...,...,...,...,...,...,...,...,...,...,...,...
89,Qwen2.5-0.5B-Instruct,model.layers.0,2.00,0.013617,0.125722,0.114438,0.132182,0.0,0.000000,0.000000,0.000000,0.000000
98,Qwen2.5-0.5B-Instruct,model.layers.8,0.00,2.691051,2.691051,0.257947,0.257947,0.0,0.000000,0.000000,0.000000,0.000000
99,Qwen2.5-0.5B-Instruct,model.layers.8,0.25,0.105772,2.691051,0.201218,0.257947,0.0,0.000000,0.000000,0.000000,0.000000
100,Qwen2.5-0.5B-Instruct,model.layers.8,1.00,0.040489,2.691051,0.171038,0.257947,0.0,0.000000,0.000000,0.000000,0.000000


In [31]:
agg_logprobs['choice_metric'] = agg_logprobs['global/proj_l2_rel'] * agg_logprobs[agg_choice_metric_col]
agg_generative['choice_metric'] = agg_generative['global/proj_l2_rel'] * agg_generative[agg_choice_metric_col]

In [32]:
metrict_to_keep_part_2 = ["diff_prob", "two_sided_tail", "lower_tail", "upper_tail", "agg_silence_score"]
# create ne dataframe with model_name, layer_name, kl, choice_metric for both logprobs and generative
agg_logprobs_choice = agg_logprobs[["model_name", "layer_name", "kl", 'choice_metric']+metrict_to_keep_part_2]
agg_generative_choice = agg_generative[["model_name", "layer_name", "kl", 'choice_metric']+metrict_to_keep_part_2]

agg_choice = agg_logprobs_choice.merge(
    agg_generative_choice,
    on=["model_name", "layer_name", "kl"],
    suffixes=("_logprob", "_generative"),
)

agg_logprobs_metrics = agg_logprobs[["model_name", "layer_name", "kl", "global/proj_l2_rel", "global/proj_l2_rel_max", "global/kl_div", "global/kl_div_max"]]

agg_choice = agg_choice.merge(
    agg_logprobs_metrics,
    on=["model_name", "layer_name", "kl"],
    how="left",
)

agg_choice

,model_name,layer_name,kl,choice_metric_logprob,diff_prob_logprob,two_sided_tail_logprob,lower_tail_logprob,upper_tail_logprob,agg_silence_score_logprob,choice_metric_generative,diff_prob_generative,two_sided_tail_generative,lower_tail_generative,upper_tail_generative,agg_silence_score_generative,global/proj_l2_rel,global/proj_l2_rel_max,global/kl_div,global/kl_div_max
0,Llama-2-7b-chat-hf,model.embed_tokens,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.662650e-07,0.0,0.000034,0.000017,0.000017,0.000017,0.051514,0.051514,3.044495,3.044495
1,Llama-2-7b-chat-hf,model.embed_tokens,0.125,0.002259,0.035947,0.027814,0.015032,0.012782,0.050978,4.388823e-04,0.0,0.019804,0.009902,0.009902,0.009902,0.044323,0.051514,0.022527,3.044495
2,Llama-2-7b-chat-hf,model.embed_tokens,0.250,0.009841,0.170436,0.105620,0.056145,0.049476,0.226581,1.581504e-03,0.0,0.072827,0.036414,0.036414,0.036414,0.043432,0.051514,0.012225,3.044495
3,Llama-2-7b-chat-hf,model.embed_tokens,0.500,0.022302,0.271115,0.524421,0.269170,0.255251,0.540285,2.453953e-03,0.0,0.118896,0.059448,0.059448,0.059448,0.041279,0.051514,0.009540,3.044495
4,Llama-2-7b-chat-hf,model.embed_tokens,1.000,0.020503,0.271112,0.524427,0.269168,0.255259,0.540280,4.045392e-03,0.0,0.213204,0.106602,0.106602,0.106602,0.037949,0.051514,0.004707,3.044495
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193,gemma-2b-it,model.norm,0.500,0.221347,0.591717,0.398053,0.186241,0.211811,0.777959,3.461930e-02,0.0,0.243350,0.121675,0.121675,0.121675,0.284523,0.306153,0.024408,4.467519
194,gemma-2b-it,model.norm,1.000,0.135835,0.306899,0.406060,0.207174,0.198886,0.514074,6.504433e-02,0.0,0.492327,0.246163,0.246163,0.246163,0.264232,0.306153,0.023484,4.467519
195,gemma-2b-it,model.norm,2.000,0.123205,0.278423,0.409244,0.203234,0.206010,0.481657,5.495070e-02,0.0,0.429648,0.214824,0.214824,0.214824,0.255794,0.306153,0.026507,4.467519
196,gemma-2b-it,model.norm,4.000,0.028970,0.144347,0.013418,0.007101,0.006317,0.151449,1.397067e-03,0.0,0.014607,0.007303,0.007303,0.007303,0.191288,0.306153,0.312896,4.467519


In [33]:
# add agg_choice_metric column that is min of choice_metric_logprob and choice_metric_generative
agg_choice["agg_choice_metric"] = agg_choice[[f'choice_metric_logprob', f'choice_metric_generative']].min(axis=1)

agg_choice['agg_silent_metric'] = agg_choice['agg_choice_metric'] / agg_choice['global/proj_l2_rel']
# agg_choice['silent_metric_logprob'] = agg_choice['choice_metric_logprob'] / agg_choice['global/proj_l2_rel']
# agg_choice['silent_metric_generative'] = agg_choice['choice_metric_generative'] / agg_choice['global/proj_l2_rel']


agg_choice
# for each group of (model_name, layer_name) take the kl value with the highest value 

,model_name,layer_name,kl,choice_metric_logprob,diff_prob_logprob,two_sided_tail_logprob,lower_tail_logprob,upper_tail_logprob,agg_silence_score_logprob,choice_metric_generative,...,two_sided_tail_generative,lower_tail_generative,upper_tail_generative,agg_silence_score_generative,global/proj_l2_rel,global/proj_l2_rel_max,global/kl_div,global/kl_div_max,agg_choice_metric,agg_silent_metric
0,Llama-2-7b-chat-hf,model.embed_tokens,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.662650e-07,...,0.000034,0.000017,0.000017,0.000017,0.051514,0.051514,3.044495,3.044495,0.000000,0.000000
1,Llama-2-7b-chat-hf,model.embed_tokens,0.125,0.002259,0.035947,0.027814,0.015032,0.012782,0.050978,4.388823e-04,...,0.019804,0.009902,0.009902,0.009902,0.044323,0.051514,0.022527,3.044495,0.000439,0.009902
2,Llama-2-7b-chat-hf,model.embed_tokens,0.250,0.009841,0.170436,0.105620,0.056145,0.049476,0.226581,1.581504e-03,...,0.072827,0.036414,0.036414,0.036414,0.043432,0.051514,0.012225,3.044495,0.001582,0.036414
3,Llama-2-7b-chat-hf,model.embed_tokens,0.500,0.022302,0.271115,0.524421,0.269170,0.255251,0.540285,2.453953e-03,...,0.118896,0.059448,0.059448,0.059448,0.041279,0.051514,0.009540,3.044495,0.002454,0.059448
4,Llama-2-7b-chat-hf,model.embed_tokens,1.000,0.020503,0.271112,0.524427,0.269168,0.255259,0.540280,4.045392e-03,...,0.213204,0.106602,0.106602,0.106602,0.037949,0.051514,0.004707,3.044495,0.004045,0.106602
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193,gemma-2b-it,model.norm,0.500,0.221347,0.591717,0.398053,0.186241,0.211811,0.777959,3.461930e-02,...,0.243350,0.121675,0.121675,0.121675,0.284523,0.306153,0.024408,4.467519,0.034619,0.121675
194,gemma-2b-it,model.norm,1.000,0.135835,0.306899,0.406060,0.207174,0.198886,0.514074,6.504433e-02,...,0.492327,0.246163,0.246163,0.246163,0.264232,0.306153,0.023484,4.467519,0.065044,0.246163
195,gemma-2b-it,model.norm,2.000,0.123205,0.278423,0.409244,0.203234,0.206010,0.481657,5.495070e-02,...,0.429648,0.214824,0.214824,0.214824,0.255794,0.306153,0.026507,4.467519,0.054951,0.214824
196,gemma-2b-it,model.norm,4.000,0.028970,0.144347,0.013418,0.007101,0.006317,0.151449,1.397067e-03,...,0.014607,0.007303,0.007303,0.007303,0.191288,0.306153,0.312896,4.467519,0.001397,0.007303


In [34]:
# Cleaner: break ties with kl first, then pick max agg_choice_metric per group via idxmax.
_tmp = agg_choice.sort_values(["model_name", "layer_name", "kl"], ascending=[True, True, False])
_idx = _tmp.groupby(["model_name", "layer_name"])['agg_choice_metric'].idxmax()

best_kl_df = (
    _tmp.loc[_idx, agg_choice.columns.tolist()]
    .sort_values(["model_name", "layer_name"])
).reset_index(drop=True)

best_kl_df

,model_name,layer_name,kl,choice_metric_logprob,diff_prob_logprob,two_sided_tail_logprob,lower_tail_logprob,upper_tail_logprob,agg_silence_score_logprob,choice_metric_generative,...,two_sided_tail_generative,lower_tail_generative,upper_tail_generative,agg_silence_score_generative,global/proj_l2_rel,global/proj_l2_rel_max,global/kl_div,global/kl_div_max,agg_choice_metric,agg_silent_metric
0,Llama-2-7b-chat-hf,model.embed_tokens,2.00,0.019349,0.271111,5.244287e-01,2.691672e-01,2.552615e-01,0.540278,0.004679,...,0.261276,0.130638,0.130638,0.130638,0.035814,0.051514,0.001979,3.044495,0.004679,0.130638
1,Llama-2-7b-chat-hf,model.layers.0,4.00,0.087784,0.126219,2.694915e-01,1.367499e-01,1.327416e-01,0.262969,0.056371,...,0.337734,0.168867,0.168867,0.168867,0.333819,0.454612,0.017882,6.306403,0.056371,0.168867
2,Llama-2-7b-chat-hf,model.layers.11,1.00,0.020754,0.274820,5.259788e-01,2.702680e-01,2.557109e-01,0.545088,0.004626,...,0.242967,0.121483,0.121483,0.121483,0.038075,0.090074,0.010618,1.407147,0.004626,0.121483
3,Llama-2-7b-chat-hf,model.layers.21,4.00,0.030540,0.271106,5.244379e-01,2.691641e-01,2.552738e-01,0.540270,0.007148,...,0.252913,0.126456,0.126456,0.126456,0.056527,0.097992,0.003810,0.795353,0.007148,0.126456
4,Llama-2-7b-chat-hf,model.norm,0.50,0.051053,0.271108,5.244349e-01,2.691651e-01,2.552699e-01,0.540273,0.114464,...,0.422650,0.211325,0.211325,1.211325,0.094495,0.101190,0.005384,0.632451,0.051053,0.540273
5,Phi-3-mini-4k-instruct,model.embed_tokens,1.00,0.022803,0.273651,5.258225e-01,2.652927e-01,2.605298e-01,0.538944,0.007145,...,0.337734,0.168867,0.168867,0.168867,0.042310,0.045035,0.004852,0.197981,0.007145,0.168867
6,Phi-3-mini-4k-instruct,model.layers.0,4.00,0.215745,0.338456,2.263398e-01,1.163869e-01,1.099529e-01,0.454843,0.112380,...,0.473848,0.236924,0.236924,0.236924,0.474329,0.596129,0.029018,1.521296,0.112380,0.236924
7,Phi-3-mini-4k-instruct,model.layers.11,2.00,0.043565,0.087013,6.835747e-02,3.562272e-02,3.273475e-02,0.122636,0.001126,...,0.006339,0.003170,0.003170,0.003170,0.355240,0.501323,0.033106,1.971606,0.001126,0.003170
8,Phi-3-mini-4k-instruct,model.layers.21,1.00,0.151439,0.192075,3.230746e-01,1.620867e-01,1.609879e-01,0.354162,0.000622,...,0.002909,0.001455,0.001455,0.001455,0.427598,0.492989,0.064960,1.699576,0.000622,0.001455
9,Phi-3-mini-4k-instruct,model.norm,0.25,0.239413,0.402593,1.256221e-01,6.374004e-02,6.188206e-02,0.466333,0.010264,...,0.039984,0.019992,0.019992,0.019992,0.513396,0.516003,0.018080,1.674800,0.010264,0.019992


In [35]:
rename_map = {
    # Experiment
    'model_name': 'Model',
    'layer_name': 'Layer',
    'kl': r'$\lambda$',
    # Experiment Results
    'agg_silent_metric': r'$\mathrm{Sil}$',
    'global/proj_l2_rel': r'$E_{L2}$',
    'global/proj_l2_rel_max': r'$\max E_{L2}$',
    'global/kl_div': r'$\mathcal{L}_{KL}$',
    'global/kl_div_max': r'$\max \mathcal{L}_{KL}$',
    # Logprob Results
    'diff_prob_logprob': r'$\mathrm{Diff}_{\text{logprob}}$',
    'lower_tail_logprob': r'$\mathrm{LTail}_{\text{logprob}}$',
    'agg_silence_score_logprob': r'$\mathrm{Sil}_{\text{logprob}}$',
    # Generative Results
    'diff_prob_generative': r'$\mathrm{Diff}_{\text{generative}}$',
    'lower_tail_generative': r'$\mathrm{LTail}_{\text{generative}}$',
    'agg_silence_score_generative': r'$\mathrm{Sil}_{\text{generative}}$',
}
mask_cols = [k for k in rename_map.keys()]
best_kl_df_renamed = best_kl_df[mask_cols].rename(columns=rename_map)

In [36]:
best_kl_df_renamed.sort_values(rename_map['agg_silent_metric'], ascending=False)

,Model,Layer,$\lambda$,$\mathrm{Sil}$,$E_{L2}$,$\max E_{L2}$,$\mathcal{L}_{KL}$,$\max \mathcal{L}_{KL}$,$\mathrm{Diff}_{\text{logprob}}$,$\mathrm{LTail}_{\text{logprob}}$,$\mathrm{Sil}_{\text{logprob}}$,$\mathrm{Diff}_{\text{generative}}$,$\mathrm{LTail}_{\text{generative}}$,$\mathrm{Sil}_{\text{generative}}$
4,Llama-2-7b-chat-hf,model.norm,0.50,0.540273,0.094495,0.101190,0.005384,0.632451,0.271108,2.691651e-01,0.540273,1.0,0.211325,1.211325
20,Qwen2.5-3B-Instruct,model.layers.24,8.00,0.534532,0.000389,0.372008,0.008496,2.623426,0.287128,2.474039e-01,0.534532,1.0,0.064806,1.064806
16,Qwen2.5-14B-Instruct,model.norm,3.00,0.281395,0.134103,0.211883,0.010046,1.228669,0.266548,2.572715e-01,0.523820,0.0,0.281395,0.281395
6,Phi-3-mini-4k-instruct,model.layers.0,4.00,0.236924,0.474329,0.596129,0.029018,1.521296,0.338456,1.163869e-01,0.454843,0.0,0.236924,0.236924
26,gemma-2b-it,model.norm,0.25,0.227834,0.294758,0.306153,0.042764,4.467519,0.563679,1.765139e-01,0.740193,0.0,0.227834,0.227834
24,gemma-2b-it,model.layers.12,0.50,0.225570,0.123231,0.167434,0.035517,5.093478,0.271147,2.087079e-01,0.479854,0.0,0.225570,0.225570
21,Qwen2.5-3B-Instruct,model.norm,0.50,0.222222,0.310078,0.328088,0.029996,1.690635,0.269252,1.672850e-01,0.436537,0.0,0.222222,0.222222
5,Phi-3-mini-4k-instruct,model.embed_tokens,1.00,0.168867,0.042310,0.045035,0.004852,0.197981,0.273651,2.652927e-01,0.538944,0.0,0.168867,0.168867
1,Llama-2-7b-chat-hf,model.layers.0,4.00,0.168867,0.333819,0.454612,0.017882,6.306403,0.126219,1.367499e-01,0.262969,0.0,0.168867,0.168867
17,Qwen2.5-3B-Instruct,model.embed_tokens,4.00,0.134852,0.151867,0.186760,0.010422,0.372046,0.287213,2.475489e-01,0.534762,0.0,0.134852,0.134852


In [ ]:
best_kl_df_renamed

In [37]:
def fmt(x, k=4):
    return f"{x:.{k}f}".rstrip("0").rstrip(".")

print(
    best_kl_df_renamed.to_latex(
        index=False,
        float_format=lambda x: fmt(x, k=4)
    )
)

\begin{tabular}{llrrrrrrrrrrrr}
\toprule
Model & Layer & $\lambda$ & $\mathrm{Sil}$ & $E_{L2}$ & $\max E_{L2}$ & $\mathcal{L}_{KL}$ & $\max \mathcal{L}_{KL}$ & $\mathrm{Diff}_{\text{logprob}}$ & $\mathrm{LTail}_{\text{logprob}}$ & $\mathrm{Sil}_{\text{logprob}}$ & $\mathrm{Diff}_{\text{generative}}$ & $\mathrm{LTail}_{\text{generative}}$ & $\mathrm{Sil}_{\text{generative}}$ \\
\midrule
Llama-2-7b-chat-hf & model.embed_tokens & 2 & 0.1306 & 0.0358 & 0.0515 & 0.002 & 3.0445 & 0.2711 & 0.2692 & 0.5403 & 0 & 0.1306 & 0.1306 \\
Llama-2-7b-chat-hf & model.layers.0 & 4 & 0.1689 & 0.3338 & 0.4546 & 0.0179 & 6.3064 & 0.1262 & 0.1367 & 0.263 & 0 & 0.1689 & 0.1689 \\
Llama-2-7b-chat-hf & model.layers.11 & 1 & 0.1215 & 0.0381 & 0.0901 & 0.0106 & 1.4071 & 0.2748 & 0.2703 & 0.5451 & 0 & 0.1215 & 0.1215 \\
Llama-2-7b-chat-hf & model.layers.21 & 4 & 0.1265 & 0.0565 & 0.098 & 0.0038 & 0.7954 & 0.2711 & 0.2692 & 0.5403 & 0 & 0.1265 & 0.1265 \\
Llama-2-7b-chat-hf & model.norm & 0.5 & 0.5403 & 0.0945 & 0.

In [38]:
def layer_order(layer_name: str) -> int:
    if ".layers." in layer_name:
        return int(layer_name.split(".")[-1])
    elif "embed_tokens" in layer_name:
        return -1
    else:
        return 1000

In [39]:
def visualize(
        df:pd.DataFrame,
        res_name: str,
        abs_diff_threshold: float,
        row:str = None,
    ):
    g = sns.catplot(
        data=df,
        x="two_sided_tail_agg",
        y="layer_name",
        kind="bar",
        col="model_name",
        row=row,
        order=sorted(df["layer_name"].unique(), key=layer_order),
    )
    g.figure.suptitle(f"Aggregated {res_name} two-sided tail values (threshold={abs_diff_threshold})", y=1.02)